# Sprint 4 — Latency and quality per configuration

Runs the same 3 golden questions through the live RAG pipeline once per configuration (model, reasoning effort, verbosity), records how long each answer took, and scores it with the three DeepEval metrics from earlier testing: **Faithfulness**, **Answer Relevancy** and **Contextual Relevancy**. Speed and quality land in one table, so a faster setup can't quietly trade away answer quality.

**Before running:**
1. Set `OPEN_AI_API_KEY` (or `OPENAI_API_KEY`) in `.env`, and have the persisted ChromaDB in `vectorstore/`.
2. Edit `CONFIGS` in section 2. That is the only cell that needs changing.
3. Run top to bottom. Each config costs 3 pipeline calls plus about 9 judge calls.

## 1. Setup

In [1]:
from pathlib import Path
from datetime import datetime
import os
import sys

from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

# The project's .env uses OPEN_AI_API_KEY, but DeepEval only reads OPENAI_API_KEY.
load_dotenv(ROOT / ".env", override=True)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPEN_AI_API_KEY", "")
if not os.environ["OPENAI_API_KEY"]:
    raise RuntimeError("No OpenAI API key found. Set OPEN_AI_API_KEY in .env.")

# Same judge timeout override as the faithfulness notebook. Must be set before deepeval is imported.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"

from deepeval.metrics import AnswerRelevancyMetric, ContextualRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase

from src.answer import _build_chat_llm
from src.config.settings import LLM_TEMPERATURE
from src.pipeline import run_query

## 2. Configurations

The only cell to edit. Each entry is one configuration to test. `None` means the parameter is not sent, so the API uses its own default; that is how the live app runs `gpt-5-mini` today.

- `reasoning_effort`: `"minimal"`, `"low"`, `"medium"`, `"high"` (which ones a model accepts depends on the model)
- `verbosity`: `"low"`, `"medium"`, `"high"`

Everything else in the pipeline (retrieval, top-k, prompt, temperature) comes from `src/config/settings.py`, so only the answering model changes between configs.

In [2]:
CONFIGS = [
    {"model": "gpt-5-mini",  "reasoning_effort": None, "verbosity": None},  # current live setup
    {"model": "gpt-6-luna",  "reasoning_effort": None, "verbosity": None},
    {"model": "gpt-6.1-sol", "reasoning_effort": None, "verbosity": None},
]

JUDGE_MODEL = "gpt-4.1"   # LLM-as-a-judge; same judge as the faithfulness notebook

RESULTS_PATH = ROOT / "tests" / "eval" / "latency_quality_results.md"


def config_label(config):
    return (
        f"{config['model']} | effort={config['reasoning_effort'] or 'default'} "
        f"| verbosity={config['verbosity'] or 'default'}"
    )


labels = [config_label(c) for c in CONFIGS]
assert len(set(labels)) == len(labels), "Two configs are identical; remove the duplicate."
print("\n".join(labels))

gpt-5-mini | effort=default | verbosity=default
gpt-6-luna | effort=default | verbosity=default
gpt-6.1-sol | effort=default | verbosity=default


## 3. Golden questions

Three questions from the golden set, copied word for word from `S4_faithfulness_eval.ipynb`. Every config answers exactly these, so their results can be compared.

In [3]:
QUESTIONS = [
    ("Q1", "What are the key safety requirements when working at height?"),
    ("Q5", "What safety considerations apply when working with or near asbestos? "),
    ("Q14", "How should risks from workplace exposure be monitored and managed?"),
]

## 4. Run each config over the questions

Latency is `run_query`'s `latency_seconds`: the whole pipeline (retrieval + answer), which is what the user waits for.

Run the first cell once. The second can be re-run: it skips (config, question) pairs already in `runs`, so one failed call doesn't cost the rest.

In [4]:
runs = {}  # (config label, question ID) -> run; run once, re-running wipes progress

In [5]:
for config in CONFIGS:
    label = config_label(config)
    llm = _build_chat_llm(
        model=config["model"],
        temperature=LLM_TEMPERATURE,
        reasoning_effort=config["reasoning_effort"],
        verbosity=config["verbosity"],
    )

    for qid, question in QUESTIONS:
        if (label, qid) in runs:
            continue
        try:
            result = run_query(question, llm=llm)
        except Exception as e:
            print(f"{label} / {qid}: PIPELINE FAILED ({type(e).__name__}: {e}), re-run this cell to retry")
            continue

        runs[(label, qid)] = {
            "config": label,
            "id": qid,
            "question": question,
            "answer": result["answer"],
            "status": result["status"],
            "error": result["error"],
            "latency_seconds": result["latency_seconds"],
            "retrieval_context": [chunk["text"] for chunk in result["chunks"]],
        }
        print(f"{label} / {qid}: status={result['status']}, {result['latency_seconds']}s")
        if result["error"]:
            print(f"  error: {result['error']}")

print(f"\nDone: {len(runs)}/{len(CONFIGS) * len(QUESTIONS)}")

gpt-5-mini | effort=default | verbosity=default / Q1: status=ok, 8.32s
gpt-5-mini | effort=default | verbosity=default / Q5: status=ok, 7.55s
gpt-5-mini | effort=default | verbosity=default / Q14: status=ok, 7.71s
gpt-6-luna | effort=default | verbosity=default / Q1: status=ok, 4.59s
gpt-6-luna | effort=default | verbosity=default / Q5: status=ok, 4.17s
gpt-6-luna | effort=default | verbosity=default / Q14: status=ok, 3.51s
gpt-6.1-sol | effort=default | verbosity=default / Q1: status=ok, 6.55s
gpt-6.1-sol | effort=default | verbosity=default / Q5: status=ok, 8.72s
gpt-6.1-sol | effort=default | verbosity=default / Q14: status=ok, 8.72s

Done: 9/9


## 5. Score with the DeepEval metrics

Answers with no retrieved context (guardrail refusals, no results, errors) are marked *not scored*: there's nothing to judge them against, so a 0 or 1 would be misleading.

Same pattern as above: run the first cell once, re-run the second to retry any judge failures.

In [6]:
scores = {}  # (config label, question ID) -> result row

In [8]:
METRICS = {
    "Faithfulness": FaithfulnessMetric,
    "Answer Relevancy": AnswerRelevancyMetric,
    "Contextual Relevancy": ContextualRelevancyMetric,
}

for key, run in runs.items():
    if key in scores:
        continue

    row = {
        "Config": run["config"],
        "ID": run["id"],
        "Status": run["status"],
        "Latency (s)": run["latency_seconds"],
        **{name: None for name in METRICS},
    }

    if run["status"] != "ok" or not run["retrieval_context"]:
        print(f"{run['config']} / {run['id']}: NOT SCORED ({run['status']})")
        scores[key] = row
        continue

    test_case = LLMTestCase(
        input=run["question"],
        actual_output=run["answer"],
        retrieval_context=run["retrieval_context"],
    )
    try:
        for name, metric_class in METRICS.items():
            metric = metric_class(model=JUDGE_MODEL, async_mode=False)  # sync avoids Jupyter event-loop clashes
            metric.measure(test_case)
            row[name] = round(metric.score, 3)  # type: ignore
    except Exception as e:
        # Left out of `scores` so the next run of this cell retries it.
        print(f"{run['config']} / {run['id']}: JUDGE FAILED ({type(e).__name__}), re-run this cell to retry")
        continue

    scores[key] = row
    print(
        f"{run['config']} / {run['id']}: {row['Latency (s)']}s, "
        + ", ".join(f"{name}={row[name]}" for name in METRICS)
    )

missing = [f"{c} / {q}" for c, q in runs if (c, q) not in scores]
print(f"\nDone: {len(scores)}/{len(runs)}" + (f", still to score: {'; '.join(missing)}" if missing else ""))

gpt-6-luna | effort=default | verbosity=default / Q5: JUDGE FAILED (LengthFinishReasonError), re-run this cell to retry


gpt-6-luna | effort=default | verbosity=default / Q14: 3.51s, Faithfulness=1.0, Answer Relevancy=1.0, Contextual Relevancy=1.0


gpt-6.1-sol | effort=default | verbosity=default / Q1: 6.55s, Faithfulness=1.0, Answer Relevancy=1.0, Contextual Relevancy=0.907


gpt-6.1-sol | effort=default | verbosity=default / Q5: 8.72s, Faithfulness=1.0, Answer Relevancy=1.0, Contextual Relevancy=0.904


KeyboardInterrupt: 

## 6. Summary

In [9]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(list(scores.values()))
metric_names = list(METRICS)

display(df[["Config", "ID", "Status", "Latency (s)", *metric_names]])

by_config = (
    df.groupby("Config", sort=False)[["Latency (s)", *metric_names]]
    .mean()
    .round(3)
    .rename(columns={"Latency (s)": "Mean latency (s)"})
)
display(by_config)

# Every config must have answered the same questions, or the comparison isn't fair.
expected_ids = {qid for qid, _ in QUESTIONS}
answered = df.groupby("Config", sort=False)["ID"].apply(set)
mismatched = [config for config, ids in answered.items() if ids != expected_ids]
if mismatched:
    print("Configs that did NOT answer every question:", mismatched)
else:
    print(f"All {len(answered)} config(s) answered the same {len(expected_ids)} questions: {sorted(expected_ids)}")

,Config,ID,Status,Latency (s),Faithfulness,Answer Relevancy,Contextual Relevancy
0,gpt-5-mini | effort=default | verbosity=default,Q1,ok,8.32,1.0,1.0,0.875
1,gpt-5-mini | effort=default | verbosity=default,Q5,ok,7.55,1.0,1.0,0.904
2,gpt-5-mini | effort=default | verbosity=default,Q14,ok,7.71,1.0,1.0,0.974
3,gpt-6-luna | effort=default | verbosity=default,Q1,ok,4.59,1.0,1.0,0.870
4,gpt-6-luna | effort=default | verbosity=default,Q14,ok,3.51,1.0,1.0,1.000
5,gpt-6.1-sol | effort=default | verbosity=default,Q1,ok,6.55,1.0,1.0,0.907
6,gpt-6.1-sol | effort=default | verbosity=default,Q5,ok,8.72,1.0,1.0,0.904


,Mean latency (s),Faithfulness,Answer Relevancy,Contextual Relevancy
Config,,,,
gpt-5-mini | effort=default | verbosity=default,7.860,1.0,1.0,0.918
gpt-6-luna | effort=default | verbosity=default,4.050,1.0,1.0,0.935
gpt-6.1-sol | effort=default | verbosity=default,7.635,1.0,1.0,0.906


Configs that did NOT answer every question: ['gpt-6-luna | effort=default | verbosity=default', 'gpt-6.1-sol | effort=default | verbosity=default']


## 7. Save results

In [ ]:
def _cell(value):
    return "–" if value is None or value != value else str(value).replace("|", "\\|")

lines = [
    "# Latency and quality results",
    "",
    f"- Run: {datetime.now():%Y-%m-%d %H:%M}",
    f"- Judge model: `{JUDGE_MODEL}`",
    f"- Questions: {', '.join(f'{qid} ({q.strip()})' for qid, q in QUESTIONS)}",
    "",
    "## Per configuration (mean)",
    "",
    "| Config | Mean latency (s) | " + " | ".join(metric_names) + " |",
    "|---|---|" + "---|" * len(metric_names),
]
for config, row in by_config.iterrows():
    lines.append(f"| {_cell(config)} | " + " | ".join(_cell(v) for v in row) + " |")

lines += [
    "",
    "## Per question",
    "",
    "| Config | ID | Status | Latency (s) | " + " | ".join(metric_names) + " |",
    "|---|---|---|---|" + "---|" * len(metric_names),
]
for _, row in df.iterrows():
    lines.append(
        f"| {_cell(row['Config'])} | {row['ID']} | {row['Status']} | {row['Latency (s)']} | "
        + " | ".join(_cell(row[name]) for name in metric_names) + " |"
    )

RESULTS_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Wrote {RESULTS_PATH.relative_to(ROOT)}")